# 데이터 로딩 및 Neo4j 임포트

이 노트북은 CSV 파일을 Neo4j 데이터베이스에 로드하는 과정을 다룹니다.


In [1]:
import pandas as pd
import sys
import os

# src 폴더를 경로에 추가
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from db_connector import Neo4jConnector


In [16]:
# Neo4j 연결 설정
URI = "bolt://localhost:7687"
USER = "neo4j"
PASSWORD = "password"  # docker-compose.yml의 NEO4J_AUTH에서 설정한 비밀번호

connector = Neo4jConnector(URI, USER, PASSWORD)
if connector.test_connection():
    print("✅ Neo4j 연결 성공!")
else:
    print("❌ Neo4j 연결 실패")


Failed to write data to connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0)))


✅ Neo4j 연결 성공!


In [ ]:
# CSV 파일 로드
import_dir = "../import"

# 데이터 확인
nodes_df = pd.read_csv(f"{import_dir}/nodes.csv")
edges_df = pd.read_csv(f"{import_dir}/edges.csv")

print(f"Nodes: {len(nodes_df)} rows")
print(f"Edges: {len(edges_df)} rows")
nodes_df.head()


## 📊 그래프 데이터 확인


In [15]:
# 1. 전체 통계 확인
node_count_query = "MATCH (n) RETURN count(n) as total_nodes"
rel_count_query = "MATCH ()-[r]->() RETURN count(r) as total_relationships"
label_count_query = "CALL db.labels() YIELD label RETURN count(label) as total_labels"

node_stats = connector.execute_query(node_count_query)
rel_stats = connector.execute_query(rel_count_query)
label_stats = connector.execute_query(label_count_query)

print("=" * 50)
print("📈 그래프 통계")
print("=" * 50)
print(f"총 노드 수: {node_stats[0]['total_nodes']:,}")
print(f"총 관계 수: {rel_stats[0]['total_relationships']:,}")
print(f"총 레이블 종류: {label_stats[0]['total_labels']}")
print("=" * 50)


📈 그래프 통계
총 노드 수: 20,000
총 관계 수: 1,081,540
총 레이블 종류: 4


In [13]:
# 2. 레이블별 노드 수 확인
label_query = """
CALL db.labels() YIELD label
CALL apoc.cypher.run('MATCH (n:' + label + ') RETURN count(n) as count', {}) YIELD value
RETURN label, value.count as count
ORDER BY count DESC
LIMIT 20
"""

try:
    labels = connector.execute_query(label_query)
    print("\n📋 레이블별 노드 수 (상위 20개)")
    print("-" * 50)
    for label_info in labels:
        print(f"{label_info['label']:30s}: {label_info['count']:,}")
except Exception as e:
    print(f"APOC를 사용할 수 없습니다. 간단한 쿼리로 확인합니다.")
    simple_label_query = """
    MATCH (n)
    RETURN labels(n)[0] as label, count(n) as count
    ORDER BY count DESC
    LIMIT 20
    """
    labels = connector.execute_query(simple_label_query)
    for label_info in labels:
        print(f"{label_info.get('label', 'Unknown'):30s}: {label_info['count']:,}")



📋 레이블별 노드 수 (상위 20개)
--------------------------------------------------
BaseNode                      : 20,000
Gene_protein                  : 12,856
Drug                          : 5,000
Anatomy                       : 2,144


In [5]:
# 3. 샘플 노드 확인
sample_nodes_query = """
MATCH (n:BaseNode)
RETURN n.node_index as index, n.name as name, labels(n) as labels, n.node_type as type
LIMIT 10
"""

sample_nodes = connector.execute_query(sample_nodes_query)
print("\n🔍 샘플 노드 (10개)")
print("-" * 80)
for node in sample_nodes:
    labels_str = ', '.join(node['labels']) if node['labels'] else 'None'
    print(f"Index: {node['index']:6d} | Name: {node['name']:30s} | Labels: {labels_str}")



🔍 샘플 노드 (10개)
--------------------------------------------------------------------------------
Index:  50499 | Name: endocytic recycling            | Labels: BaseNode, Biological_process
Index:  16574 | Name: Gonadorelin                    | Labels: BaseNode, Drug
Index:  50500 | Name: late endosome to vacuole transport via multivesicular body sorting pathway | Labels: BaseNode, Biological_process
Index:  50501 | Name: establishment of protein localization to mitochondrial membrane | Labels: BaseNode, Biological_process
Index: 101867 | Name: lincomycin biosynthetic process | Labels: BaseNode, Biological_process
Index:  16906 | Name: N-(2,3,4,5,6-Pentaflouro-Benzyl)-4-Sulfamoyl-Benzamide | Labels: BaseNode, Drug
Index: 101868 | Name: ectoine biosynthetic process   | Labels: BaseNode, Biological_process
Index:  15000 | Name: Lapatinib                      | Labels: BaseNode, Drug
Index: 101869 | Name: limonene biosynthetic process  | Labels: BaseNode, Biological_process
Index: 120466 | 

In [12]:
# 4. 관계 타입별 통계
# 먼저 총 관계 타입 개수 확인
total_rel_types_query = """
MATCH ()-[r]->()
RETURN count(DISTINCT type(r)) as total_types
"""
total_types = connector.execute_query(total_rel_types_query)
print(f"\n📊 총 관계 타입 종류: {total_types[0]['total_types']}개")

# 모든 관계 타입 확인 (LIMIT 제거)
relationship_query = """
MATCH ()-[r]->()
RETURN type(r) as relationship_type, count(r) as count
ORDER BY count DESC
"""

relationships = connector.execute_query(relationship_query)
print(f"\n🔗 관계 타입별 통계 (총 {len(relationships)}개)")
print("-" * 60)
for rel in relationships:
    print(f"{rel['relationship_type']:40s}: {rel['count']:,}")

# CSV 파일과 비교
print("\n" + "=" * 60)
print("📝 참고: edges.csv에는 18가지 관계 타입이 있습니다.")
print("   데이터 로딩이 완전하지 않을 수 있습니다.")
print("=" * 60)



📊 총 관계 타입 종류: 6개

🔗 관계 타입별 통계 (총 6개)
------------------------------------------------------------

📝 참고: edges.csv에는 18가지 관계 타입이 있습니다.
   데이터 로딩이 완전하지 않을 수 있습니다.


In [8]:
# 5. 샘플 그래프 구조 확인 (노드와 관계 함께)
sample_graph_query = """
MATCH (a:BaseNode)-[r]->(b:BaseNode)
RETURN a.name as from_node, type(r) as relationship, b.name as to_node
LIMIT 10
"""

sample_graph = connector.execute_query(sample_graph_query)
print("\n🌐 샘플 그래프 구조 (10개)")
print("-" * 100)
for edge in sample_graph:
    print(f"{edge['from_node']:30s} --[{edge['relationship']:20s}]--> {edge['to_node']:30s}")



🌐 샘플 그래프 구조 (10개)
----------------------------------------------------------------------------------------------------
fast endocytic recycling       --[PARENT-CHILD        ]--> endocytic recycling           
slow endocytic recycling       --[PARENT-CHILD        ]--> endocytic recycling           
endosome to plasma membrane protein transport --[PARENT-CHILD        ]--> endocytic recycling           
Corifollitropin alfa           --[SYNERGISTIC_INTERACTION]--> Gonadorelin                   
Capromab pendetide             --[SYNERGISTIC_INTERACTION]--> Gonadorelin                   
protein transport to vacuole involved in ubiquitin-dependent protein catabolic process via the multivesicular body sorting pathway --[PARENT-CHILD        ]--> late endosome to vacuole transport via multivesicular body sorting pathway
establishment of protein localization to mitochondrial membrane involved in mitochondrial fission --[PARENT-CHILD        ]--> establishment of protein localization to mitochon

## 🌐 Neo4j Browser로 시각화하기

웹 브라우저에서 Neo4j Browser를 열어 그래프를 시각적으로 확인할 수 있습니다:

1. **브라우저에서 접속**: http://localhost:7474
2. **로그인 정보**:
   - Username: `neo4j`
   - Password: `password`
3. **유용한 쿼리 예시**:


### Neo4j Browser에서 실행할 쿼리 예시:

```cypher
// 1. 전체 그래프 구조 확인 (노드 100개, 관계 200개 제한)
MATCH (n)-[r]->(m)
RETURN n, r, m
LIMIT 200

// 2. 특정 노드와 연결된 모든 노드 확인
MATCH (n:BaseNode {name: '노드이름'})-[r]-(connected)
RETURN n, r, connected

// 3. 가장 많은 연결을 가진 노드 찾기
MATCH (n:BaseNode)-[r]-()
RETURN n.name as node_name, count(r) as connection_count
ORDER BY connection_count DESC
LIMIT 20

// 4. 질병(Disease) 노드만 확인
MATCH (n)
WHERE 'Disease' IN labels(n)
RETURN n
LIMIT 50

// 5. 약물(Drug) 노드만 확인
MATCH (n)
WHERE 'Drug' IN labels(n)
RETURN n
LIMIT 50
```


In [ ]:
# 연결 종료
connector.close()
print("✅ 연결이 종료되었습니다.")


In [10]:
# 추가: CSV 파일의 display_relation 종류 확인
import pandas as pd
edges_df = pd.read_csv("../import/edges.csv")
print("\n📋 CSV 파일의 display_relation 종류:")
print("-" * 60)
display_relations = edges_df['display_relation'].value_counts()
print(f"총 {len(display_relations)}가지 관계 타입")
print("\n상위 10개:")
for rel_type, count in display_relations.head(10).items():
    # Neo4j에서 변환될 형태로 표시
    neo4j_format = rel_type.upper().replace(' ', '_')
    print(f"  {rel_type:30s} ({neo4j_format:30s}): {count:,}")



📋 CSV 파일의 display_relation 종류:
------------------------------------------------------------
총 18가지 관계 타입

상위 10개:
  expression present             (EXPRESSION_PRESENT            ): 3,036,406
  synergistic interaction        (SYNERGISTIC_INTERACTION       ): 2,672,628
  interacts with                 (INTERACTS_WITH                ): 686,550
  ppi                            (PPI                           ): 642,150
  phenotype present              (PHENOTYPE_PRESENT             ): 300,634
  parent-child                   (PARENT-CHILD                  ): 281,744
  associated with                (ASSOCIATED_WITH               ): 167,482
  side effect                    (SIDE_EFFECT                   ): 129,568
  contraindication               (CONTRAINDICATION              ): 61,350
  expression absent              (EXPRESSION_ABSENT             ): 39,774
